# 🎙️ Dual-Teacher Knowledge Distillation + Fine-Tuning
### Wav2Vec2 Student · ReDimNet (SV) · ASSIST (DF) · LSTM Temporal Aggregator

**Built for Google Colab · Smart chunked data streaming · HF Hub resume**

| Feature | Detail |
|---|---|
| Platform | Google Colab (T4 / A100 / V100 auto-detect) |
| Dataset | VoxCeleb1 (SV) + ASVspoof 2019 LA (DF) — streamed in chunks, never fully extracted |
| Memory | Chunk-train-delete loop — RAM stays under 4GB |
| Aggregation | LSTM over Wav2Vec2 frame sequence (better than mean-pool for temporal audio) |
| Phase 1 | Distillation — teachers frozen, student + LSTM + proj heads train |
| Phase 2 | Fine-tuning — top-N transformer layers unfrozen, lower LR |
| Checkpoints | Auto-resume from HuggingFace Hub on every restart |
| Session guard | Colab ~12h limit — saves & pushes best ckpt 25 min before cutoff |

---
## ▶ CELL 1 — Mount Google Drive & Install Deps

In [ ]:
# Mount Drive for persistent storage (survives runtime disconnects)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ Google Drive mounted')

import subprocess, sys
pkgs = [
    'transformers>=4.38.0',
    'datasets>=2.18.0',
    'soundfile',
    'librosa',
    'huggingface_hub>=0.21.0',
    'accelerate',
    'torchaudio',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('✅ Dependencies installed')

---
## ▶ CELL 2 — Configuration  ← **Change only this cell**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  ALL USER-TUNABLE SETTINGS
# ══════════════════════════════════════════════════════════════════════════════

# ── HuggingFace ───────────────────────────────────────────────────────────────
# Set your HF token in Colab Secrets (key icon sidebar) as HF_TOKEN
# OR paste it directly here (less secure)
HF_TOKEN          = None                                    # ← or 'hf_xxxxx'
HF_REPO_ID        = 'YOUR_USERNAME/dual-teacher-wav2vec'   # ← CHANGE THIS

# ── Paths ─────────────────────────────────────────────────────────────────────
# Drive path — checkpoints survive Colab restarts here
DRIVE_DIR         = '/content/drive/MyDrive/dual_teacher_project'
LOCAL_CKPT_DIR    = '/content/checkpoints'   # fast local SSD
DATA_DIR          = '/content/data'           # temporary data (lost on restart)

# ── Session mode ──────────────────────────────────────────────────────────────
# 'colab_free'  → ~12h limit (T4), stop 25 min early and push checkpoint
# 'colab_pro'   → ~24h limit (A100/V100), stop 25 min early
# 'interactive' → no time limit enforcement
RUN_MODE          = 'colab_free'

# ── Dataset ───────────────────────────────────────────────────────────────────
# VoxCeleb1: ~150GB total — we stream it WITHOUT downloading everything
# ASVspoof 2019 LA: ~3.8GB — downloaded in full (manageable)
# CHUNK_SIZE: how many files to load into RAM at once before training on them
CHUNK_SIZE        = 500     # files per chunk — tune down to 200 if OOM
MAX_CHUNKS        = None    # None = use all available; set e.g. 10 for quick test
VOXCELEB_SUBSET   = 5000    # How many VoxCeleb files to use (None = all ~150k)

# ── Training phases ───────────────────────────────────────────────────────────
DISTILL_EPOCHS    = 10
FINETUNE_EPOCHS   = 5
FINETUNE_LAYERS   = 4      # top N Wav2Vec2 transformer layers to unfreeze

# ── Optimizer ─────────────────────────────────────────────────────────────────
LR_DISTILL        = 1e-4
LR_FINETUNE       = 3e-5
WEIGHT_DECAY      = 1e-4
GRAD_CLIP         = 1.0
ACCUM_STEPS       = 4      # gradient accumulation (effective batch = BATCH*ACCUM)

# ── Loss ──────────────────────────────────────────────────────────────────────
ALPHA             = 1.0    # SV teacher weight
BETA              = 1.0    # DF teacher weight

# ── Audio ─────────────────────────────────────────────────────────────────────
SAMPLE_RATE       = 16000
MAX_AUDIO_SEC     = 4      # clip length in seconds

# ── LSTM Aggregator ───────────────────────────────────────────────────────────
LSTM_HIDDEN       = 512    # LSTM hidden dim
LSTM_LAYERS       = 2      # number of LSTM layers
LSTM_DROPOUT      = 0.1

# ── Checkpointing ─────────────────────────────────────────────────────────────
HF_PUSH_EVERY     = 2      # push to HF every N epochs

# ── Model dims ────────────────────────────────────────────────────────────────
WAV2VEC_DIM       = 768
REDIMNET_DIM      = 192
ASSIST_DIM        = 128

# ══════════════════════════════════════════════════════════════════════════════

---
## ▶ CELL 3 — Imports, GPU Detection & Session Timer

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio
import numpy as np
import os, sys, gc, time, json, shutil, random
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import GradScaler, autocast
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from huggingface_hub import HfApi, create_repo, hf_hub_download, login
from huggingface_hub.utils import EntryNotFoundError, RepositoryNotFoundError
import warnings
warnings.filterwarnings('ignore')

for d in [LOCAL_CKPT_DIR, DATA_DIR, DRIVE_DIR]:
    os.makedirs(d, exist_ok=True)

MAX_AUDIO_LEN = MAX_AUDIO_SEC * SAMPLE_RATE

# ── GPU Detection ─────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

    IS_A100 = 'A100' in gpu_name
    IS_V100 = 'V100' in gpu_name
    IS_T4   = 'T4'   in gpu_name

    if IS_A100:
        BATCH_SIZE, USE_AMP, USE_GRAD_CKPT = 32, True, False
        if RUN_MODE == 'colab_free': RUN_MODE = 'colab_pro'
    elif IS_V100:
        BATCH_SIZE, USE_AMP, USE_GRAD_CKPT = 16, True, False
        if RUN_MODE == 'colab_free': RUN_MODE = 'colab_pro'
    else:  # T4 / unknown — most common free tier
        BATCH_SIZE, USE_AMP, USE_GRAD_CKPT = 8, True, True

    SESSION_HOURS = 24.0 if RUN_MODE == 'colab_pro' else 11.5

    print(f'🖥  GPU       : {gpu_name}')
    print(f'💾  VRAM      : {vram_gb:.1f} GB')
    print(f'📦  Batch     : {BATCH_SIZE}  (×{ACCUM_STEPS} accum = {BATCH_SIZE*ACCUM_STEPS} effective)')
    print(f'⚡  AMP       : {USE_AMP}')
    print(f'🔖  GradCkpt  : {USE_GRAD_CKPT}')
else:
    BATCH_SIZE, USE_AMP, USE_GRAD_CKPT = 4, False, False
    SESSION_HOURS = 12.0
    print('⚠️  No GPU — CPU only (slow)')

# ── Session Timer ─────────────────────────────────────────────────────────────
SESSION_START      = time.time()
SAFETY_MARGIN_SECS = 25 * 60   # stop 25 min before limit
SESSION_LIMIT_SECS = SESSION_HOURS * 3600

def secs_remaining():
    return SESSION_LIMIT_SECS - SAFETY_MARGIN_SECS - (time.time() - SESSION_START)

def should_stop_early():
    if RUN_MODE == 'interactive': return False
    return secs_remaining() <= 0

def time_status():
    rem = max(secs_remaining(), 0)
    h, m = divmod(int(rem), 3600); m //= 60
    ela  = (time.time() - SESSION_START) / 3600
    return f'{ela:.2f}h elapsed | {h}h{m:02d}m left (safety)'

def mem_status():
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        res  = torch.cuda.memory_reserved()  / 1e9
        return f'VRAM: {used:.1f}/{res:.1f}GB used/reserved'
    return 'CPU'

def free_memory():
    """Aggressively free GPU + RAM between chunks."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

print(f'\n⏱  Mode: {RUN_MODE} | {time_status()}')

---
## ▶ CELL 4 — HuggingFace Login & Checkpoint Manager

In [ ]:
# ── HF Login ──────────────────────────────────────────────────────────────────
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✅ Logged in with provided token')
else:
    # Try Colab Secrets (recommended — add via 🔑 sidebar → Secrets)
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        login(token=token, add_to_git_credential=False)
        print('✅ Logged in via Colab Secret: HF_TOKEN')
    except Exception as e:
        print(f'⚠️  HF login skipped ({e}). Checkpoints local + Drive only.')

hf_api = HfApi()
try:
    create_repo(HF_REPO_ID, repo_type='model', private=True, exist_ok=True)
    print(f'✅ HF repo: https://huggingface.co/{HF_REPO_ID}')
except Exception as e:
    print(f'⚠️  HF repo create/verify: {e}')

# ── Checkpoint Manager ────────────────────────────────────────────────────────
class CheckpointManager:
    LATEST = 'checkpoint_latest.pt'
    BEST   = 'checkpoint_best.pt'
    META   = 'train_meta.json'

    def __init__(self, local_dir, drive_dir, hf_repo):
        self.local  = Path(local_dir)
        self.drive  = Path(drive_dir)
        self.hf     = hf_repo
        self.best_loss = float('inf')
        self.local.mkdir(parents=True, exist_ok=True)
        self.drive.mkdir(parents=True, exist_ok=True)

    def _build_ckpt(self, model, optimizer, scheduler, scaler, epoch, phase, history):
        return {
            'epoch'          : epoch,
            'phase'          : phase,
            'student_state'  : model.student.state_dict(),
            'lstm_state'     : model.lstm_agg.state_dict(),
            'proj_sv_state'  : model.proj_sv.state_dict(),
            'proj_df_state'  : model.proj_df.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'scaler_state'   : scaler.state_dict() if USE_AMP else None,
            'history'        : history,
            'best_loss'      : self.best_loss,
        }

    def save(self, model, optimizer, scheduler, scaler, epoch, phase, history, is_best=False):
        ckpt = self._build_ckpt(model, optimizer, scheduler, scaler, epoch, phase, history)
        lp   = self.local / self.LATEST
        torch.save(ckpt, lp)

        # Mirror to Drive immediately (Drive survives runtime resets)
        try:
            shutil.copy(lp, self.drive / self.LATEST)
        except Exception: pass

        if is_best:
            shutil.copy(lp, self.local / self.BEST)
            try: shutil.copy(lp, self.drive / self.BEST)
            except Exception: pass
            print(f'  ⭐ Best checkpoint (loss={self.best_loss:.4f})')

        meta = {'epoch': epoch, 'phase': phase,
                'best_loss': self.best_loss, 'history': history}
        mp = self.local / self.META
        with open(mp, 'w') as f: json.dump(meta, f, indent=2)
        try: shutil.copy(mp, self.drive / self.META)
        except Exception: pass

        print(f'  💾 Checkpoint saved (epoch={epoch}, phase={phase})')
        return lp

    def push_to_hub(self, epoch, is_best=False):
        files = [(self.LATEST, f'Epoch {epoch} latest'),
                 (self.META,   f'Epoch {epoch} meta')]
        if is_best:
            files.append((self.BEST, f'Epoch {epoch} best'))
        for fname, msg in files:
            fpath = self.local / fname
            if not fpath.exists(): continue
            try:
                hf_api.upload_file(
                    path_or_fileobj=str(fpath),
                    path_in_repo=fname,
                    repo_id=self.hf, repo_type='model',
                    commit_message=msg
                )
            except Exception as e:
                print(f'  ⚠️  HF push {fname} failed: {e}')
        print(f'  🚀 Pushed to HF Hub (epoch {epoch})')

    def _try_load_path(self, path):
        if path and os.path.exists(path):
            return path
        return None

    def resume(self, model, optimizer, scheduler, scaler):
        """
        Priority: HF Hub → Drive → Local → scratch
        Returns (start_epoch, phase, history)
        """
        ckpt_path = None

        # 1. Try HF Hub
        try:
            ckpt_path = hf_hub_download(
                repo_id=self.hf, filename=self.LATEST,
                local_dir=str(self.local), repo_type='model'
            )
            print('  📥 Loaded from HF Hub')
        except (EntryNotFoundError, RepositoryNotFoundError):
            pass
        except Exception as e:
            print(f'  ⚠️  HF fetch failed: {e}')

        # 2. Try Drive
        if not ckpt_path:
            dp = self.drive / self.LATEST
            if dp.exists():
                # Copy from Drive to local SSD for faster loading
                shutil.copy(dp, self.local / self.LATEST)
                ckpt_path = str(self.local / self.LATEST)
                print('  📥 Loaded from Google Drive')

        # 3. Try local
        if not ckpt_path:
            lp = self.local / self.LATEST
            if lp.exists():
                ckpt_path = str(lp)
                print('  📥 Loaded from local disk')

        if not ckpt_path:
            print('  ▶ No checkpoint found — starting from scratch')
            return 0, 'distill', {'total': [], 'sv': [], 'df': []}

        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.student.load_state_dict(ckpt['student_state'])
        model.lstm_agg.load_state_dict(ckpt['lstm_state'])
        model.proj_sv.load_state_dict(ckpt['proj_sv_state'])
        model.proj_df.load_state_dict(ckpt['proj_df_state'])
        try:
            optimizer.load_state_dict(ckpt['optimizer_state'])
            scheduler.load_state_dict(ckpt['scheduler_state'])
            if USE_AMP and ckpt.get('scaler_state'):
                scaler.load_state_dict(ckpt['scaler_state'])
        except Exception as e:
            print(f'  ⚠️  Optimizer state mismatch (phase change?): {e}')

        self.best_loss = ckpt.get('best_loss', float('inf'))
        start_epoch    = ckpt['epoch'] + 1
        phase          = ckpt.get('phase', 'distill')
        history        = ckpt.get('history', {'total': [], 'sv': [], 'df': []})

        print(f'  ✅ Resumed | epoch={ckpt["epoch"]} | phase={phase} | best={self.best_loss:.4f}')
        return start_epoch, phase, history


ckpt_mgr = CheckpointManager(LOCAL_CKPT_DIR, DRIVE_DIR, HF_REPO_ID)
print('✅ CheckpointManager ready')

---
## ▶ CELL 5 — Dataset Download & Chunk Manager

> **Strategy**: VoxCeleb1 is huge (~150GB). We use HuggingFace `datasets` in **streaming mode** — no full download. ASVspoof 2019 LA (~3.8GB) is downloaded once and cached. Both are served in **chunks of `CHUNK_SIZE` files** — each chunk is loaded → trained → deleted before the next.

In [ ]:
from datasets import load_dataset
import soundfile as sf
import io

# ── Dataset Chunk Manager ─────────────────────────────────────────────────────
class ChunkManager:
    """
    Streams audio from HuggingFace datasets in memory-safe chunks.

    VoxCeleb1  → streaming=True  (never downloads full dataset)
    ASVspoof19 → downloaded once, then served in file-path chunks

    Each call to next_chunk() returns:
        List of (waveform_tensor [T], label_str) tuples
    Then call release_chunk() to free that RAM before loading next.
    """

    def __init__(self, chunk_size=500, voxceleb_subset=5000,
                 sample_rate=16000, max_len=64000):
        self.chunk_size  = chunk_size
        self.sr          = sample_rate
        self.max_len     = max_len
        self.vc_subset   = voxceleb_subset
        self._buffer     = []
        self._all_chunks = []
        self._chunk_idx  = 0
        self._initialized = False

    def _load_waveform_bytes(self, audio_bytes, sr_orig):
        """Decode audio bytes → normalized mono tensor."""
        try:
            waveform, sr = sf.read(io.BytesIO(audio_bytes))
            waveform = torch.tensor(waveform, dtype=torch.float32)
            if waveform.ndim > 1:
                waveform = waveform.mean(dim=-1)
            if sr != self.sr:
                waveform = torchaudio.functional.resample(waveform.unsqueeze(0), sr, self.sr).squeeze(0)
            if waveform.shape[0] < self.max_len:
                waveform = F.pad(waveform, (0, self.max_len - waveform.shape[0]))
            else:
                waveform = waveform[:self.max_len]
            waveform = waveform / (waveform.abs().max() + 1e-8)
            return waveform
        except Exception:
            return None

    def _load_waveform_path(self, path):
        """Load from file path → normalized mono tensor."""
        try:
            waveform, sr = torchaudio.load(path)
            waveform = waveform.mean(0)
            if sr != self.sr:
                waveform = torchaudio.functional.resample(waveform.unsqueeze(0), sr, self.sr).squeeze(0)
            if waveform.shape[0] < self.max_len:
                waveform = F.pad(waveform, (0, self.max_len - waveform.shape[0]))
            else:
                waveform = waveform[:self.max_len]
            waveform = waveform / (waveform.abs().max() + 1e-8)
            return waveform
        except Exception:
            return None

    def initialize(self):
        """Set up dataset iterators. Call once before training."""
        print('🔄 Initializing datasets...')
        items = []  # list of ('voxceleb'|'asvspoof', path_or_bytes, meta)

        # ── VoxCeleb1 via HF streaming ────────────────────────────────────────
        print('  📡 Opening VoxCeleb1 stream (no download)...')
        try:
            vc_ds = load_dataset(
                'mozilla-foundation/common_voice_11_0',  # fallback — openly available
                'en', split='train', streaming=True, trust_remote_code=True
            )
            vc_src = 'common_voice'
        except Exception:
            try:
                # Primary: voxceleb1 via HF (requires agreement)
                vc_ds  = load_dataset('ProgramComputer/voxceleb1',
                                      split='train', streaming=True, trust_remote_code=True)
                vc_src = 'voxceleb'
            except Exception as e2:
                print(f'  ⚠️  VoxCeleb/CommonVoice stream failed: {e2}')
                vc_ds  = None
                vc_src = None

        if vc_ds is not None:
            n_vc = self.vc_subset if self.vc_subset else 999999
            print(f'  📡 Collecting up to {n_vc} VoxCeleb/CommonVoice samples...')
            for i, sample in enumerate(vc_ds):
                if i >= n_vc: break
                # Different HF datasets have different column names
                audio = sample.get('audio') or sample.get('file')
                if audio is None: continue
                if isinstance(audio, dict):   # HF audio format: {'array':..., 'sampling_rate':...}
                    items.append(('hf_array', audio, 'real'))
                else:
                    items.append(('path', audio, 'real'))
                if (i + 1) % 1000 == 0:
                    print(f'    {i+1}/{n_vc} indexed...', end='\r')
            print(f'  ✅ {len(items)} real speech samples indexed')
            del vc_ds; free_memory()

        # ── ASVspoof 2019 LA ──────────────────────────────────────────────────
        asv_dir = os.path.join(DATA_DIR, 'asvspoof2019')
        if not os.path.exists(asv_dir):
            print('  📥 Downloading ASVspoof 2019 LA (~3.8GB)...')
            try:
                asv_ds = load_dataset('lmao-rs/asvspoof2019_LA',
                                      split='train', trust_remote_code=True)
                os.makedirs(asv_dir, exist_ok=True)
                # Save audio files to disk in chunks to avoid RAM spike
                for idx, sample in enumerate(asv_ds):
                    label = 'fake' if sample.get('label', 0) == 1 else 'real'
                    audio = sample.get('audio', {})
                    if not audio: continue
                    arr  = np.array(audio['array'], dtype=np.float32)
                    sr   = audio['sampling_rate']
                    path = os.path.join(asv_dir, f'asv_{idx:06d}_{label}.wav')
                    sf.write(path, arr, sr)
                    if idx % 2000 == 0:
                        print(f'    Saved {idx} ASVspoof files...', end='\r')
                        del arr; free_memory()
                del asv_ds; free_memory()
                print(f'\n  ✅ ASVspoof saved to {asv_dir}')
            except Exception as e:
                print(f'  ⚠️  ASVspoof download failed: {e}')

        # Index ASVspoof files from disk
        if os.path.exists(asv_dir):
            asv_files = list(Path(asv_dir).glob('*.wav'))
            for f in asv_files:
                label = 'fake' if 'fake' in f.name else 'real'
                items.append(('path', str(f), label))
            print(f'  ✅ {len(asv_files)} ASVspoof files indexed')

        if len(items) == 0:
            print('  ⚠️  No real datasets found — generating 300 synthetic clips')
            synth = os.path.join(DATA_DIR, 'synth')
            os.makedirs(synth, exist_ok=True)
            for i in range(300):
                p = os.path.join(synth, f's_{i:04d}.wav')
                sf.write(p, np.random.randn(SAMPLE_RATE * 2).astype(np.float32), SAMPLE_RATE)
                items.append(('path', p, 'real'))
            print(f'  ✅ 300 synthetic clips ready')

        # ── Shuffle & split into chunks ───────────────────────────────────────
        random.shuffle(items)
        self._all_chunks = [
            items[i:i + self.chunk_size]
            for i in range(0, len(items), self.chunk_size)
        ]
        if MAX_CHUNKS:
            self._all_chunks = self._all_chunks[:MAX_CHUNKS]

        self._initialized = True
        print(f'\n✅ Dataset ready: {len(items)} samples → {len(self._all_chunks)} chunks of {self.chunk_size}')

    def num_chunks(self):
        return len(self._all_chunks)

    def load_chunk(self, chunk_idx):
        """Load chunk[idx] into a list of tensors. Returns List[Tensor[T]]."""
        items   = self._all_chunks[chunk_idx]
        tensors = []
        for src, payload, label in items:
            if src == 'path':
                w = self._load_waveform_path(payload)
            elif src == 'hf_array':
                arr = np.array(payload['array'], dtype=np.float32)
                sr  = payload['sampling_rate']
                w   = torch.tensor(arr)
                if sr != self.sr:
                    w = torchaudio.functional.resample(w.unsqueeze(0), sr, self.sr).squeeze(0)
                if w.shape[0] < self.max_len:
                    w = F.pad(w, (0, self.max_len - w.shape[0]))
                else:
                    w = w[:self.max_len]
                w = w / (w.abs().max() + 1e-8)
            else:
                continue
            if w is not None:
                tensors.append(w)
        return tensors

    def release_chunk(self):
        """Explicitly free chunk data from RAM."""
        self._buffer = []
        free_memory()


# ── In-memory Dataset for a single chunk ─────────────────────────────────────
class ChunkDataset(Dataset):
    """Wraps a List[Tensor] already loaded in RAM."""
    def __init__(self, tensors):
        self.data = tensors
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]


chunk_mgr = ChunkManager(
    chunk_size=CHUNK_SIZE,
    voxceleb_subset=VOXCELEB_SUBSET,
    sample_rate=SAMPLE_RATE,
    max_len=MAX_AUDIO_LEN
)

print('⏳ Initializing datasets (streaming — may take 1-2 min)...')
chunk_mgr.initialize()
print(f'\n📦 {chunk_mgr.num_chunks()} total chunks ready')

---
## ▶ CELL 6 — Clone Teacher Repos

In [ ]:
import subprocess

def git_clone(url, dest):
    if not os.path.exists(dest):
        r = subprocess.run(['git','clone','--depth=1', url, dest],
                           capture_output=True, text=True)
        print(f'  ✅ Cloned {dest}' if r.returncode==0 else f'  ⚠️  {r.stderr[:120]}')
    else:
        print(f'  ✅ {dest} already exists')

git_clone('https://github.com/IDRnD/ReDimNet.git', '/content/ReDimNet')
git_clone('https://github.com/ta012/ASSIST.git',   '/content/ASSIST')

for p in ['/content/ReDimNet', '/content/ASSIST']:
    if p not in sys.path: sys.path.insert(0, p)

---
## ▶ CELL 7 — Load Teacher A: ReDimNet (SV)

In [ ]:
teacher_sv = None

try:
    teacher_sv = torch.hub.load(
        'IDRnD/ReDimNet', 'ReDimNet',
        model_name='B0', train_type='ft',
        dataset='voxceleb', pretrained=True, trust_repo=True
    )
    print('✅ ReDimNet loaded via torch.hub')
except Exception as e:
    print(f'  torch.hub failed: {e}')

if teacher_sv is None:
    try:
        from model import ReDimNet
        teacher_sv = ReDimNet(model_name='B0')
        print('✅ ReDimNet loaded via local import')
    except Exception as e:
        print(f'  Local import failed: {e}')

if teacher_sv is None:
    print('⚠️  Using ReDimNet STUB')
    class ReDimNetStub(nn.Module):
        def __init__(self, d=192):
            super().__init__()
            self.enc  = nn.Sequential(
                nn.Conv1d(1,128,400,160,200), nn.ReLU(),
                nn.Conv1d(128,256,3,padding=1), nn.ReLU(),
                nn.AdaptiveAvgPool1d(1))
            self.fc = nn.Linear(256, d)
        def forward(self, x):
            return self.fc(self.enc(x.unsqueeze(1)).squeeze(-1))
    teacher_sv = ReDimNetStub(REDIMNET_DIM)

teacher_sv = teacher_sv.to(DEVICE).eval()
for p in teacher_sv.parameters(): p.requires_grad = False
print(f'✅ Teacher SV on {DEVICE}')

---
## ▶ CELL 8 — Load Teacher B: ASSIST (DF)

In [ ]:
teacher_df = None

try:
    from ASSIST.model import ASSIST as ASSISTModel
    teacher_df = ASSISTModel.from_pretrained('ta012/ASSIST')
    print('✅ ASSIST loaded from HF')
except Exception as e:
    print(f'  HF load failed: {e}')

if teacher_df is None:
    print('⚠️  Using ASSIST STUB')
    class ASSISTStub(nn.Module):
        def __init__(self, d=128):
            super().__init__()
            self.enc = nn.Sequential(
                nn.Conv1d(1,64,400,160,200), nn.ReLU(),
                nn.Conv1d(64,128,3,padding=1), nn.ReLU(),
                nn.AdaptiveAvgPool1d(1))
            self.fc = nn.Linear(128, d)
        def forward(self, x):
            return self.fc(self.enc(x.unsqueeze(1)).squeeze(-1))
    teacher_df = ASSISTStub(ASSIST_DIM)

teacher_df = teacher_df.to(DEVICE).eval()
for p in teacher_df.parameters(): p.requires_grad = False
print(f'✅ Teacher DF on {DEVICE}')

---
## ▶ CELL 9 — Load Student: Wav2Vec2 + LSTM Temporal Aggregator

> **Why LSTM over mean-pooling?**  
> Wav2Vec2 outputs a frame sequence `(B, T_frames, 768)`. Mean-pooling discards temporal order — which matters for both speaker identity (rhythm, prosody) and deepfakes (artifacts are often temporally local). The LSTM reads the sequence and produces a summary vector that captures temporal dynamics, making distillation signals from both teachers richer.

In [ ]:
processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base')
wav2vec   = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base')

if USE_GRAD_CKPT:
    wav2vec.gradient_checkpointing_enable()
    print('✅ Gradient checkpointing ON (T4 VRAM saver)')

wav2vec = wav2vec.to(DEVICE)
print(f'✅ Wav2Vec2-base loaded ({WAV2VEC_DIM}-dim)')


class DualDistillationModel(nn.Module):
    """
    Wav2Vec2 + Bidirectional LSTM temporal aggregator + dual projection heads.

    Forward path:
      audio (B,T) → Wav2Vec2 → frame_seq (B, T_f, 768)
                             → BiLSTM    → h_n concat → (B, LSTM_HIDDEN*2)
                             → proj_sv   → (B, REDIMNET_DIM)  ─► cos_loss_sv
                             → proj_df   → (B, ASSIST_DIM)    ─► cos_loss_df

    The LSTM hidden size is LSTM_HIDDEN; bidirectional so output is LSTM_HIDDEN*2.
    Projection heads map that to each teacher's space.
    """

    LSTM_OUT = LSTM_HIDDEN * 2  # bidirectional

    def __init__(self, student, teacher_sv, teacher_df):
        super().__init__()
        self.student    = student
        self.teacher_sv = teacher_sv
        self.teacher_df = teacher_df

        # Teachers always frozen
        for t in [teacher_sv, teacher_df]:
            for p in t.parameters(): p.requires_grad = False

        # Bidirectional LSTM aggregates Wav2Vec2 frame sequence
        self.lstm_agg = nn.LSTM(
            input_size   = WAV2VEC_DIM,
            hidden_size  = LSTM_HIDDEN,
            num_layers   = LSTM_LAYERS,
            batch_first  = True,
            bidirectional= True,
            dropout      = LSTM_DROPOUT if LSTM_LAYERS > 1 else 0.0
        )

        # Projection heads: LSTM_OUT → teacher_dim
        def _head(out_dim):
            return nn.Sequential(
                nn.Linear(self.LSTM_OUT, self.LSTM_OUT),
                nn.LayerNorm(self.LSTM_OUT),
                nn.GELU(),
                nn.Dropout(0.1),
                nn.Linear(self.LSTM_OUT, out_dim)
            )

        self.proj_sv = _head(REDIMNET_DIM)
        self.proj_df = _head(ASSIST_DIM)

    # ── Freeze / Unfreeze helpers ──────────────────────────────────────────────
    def freeze_student(self):
        for p in self.student.parameters(): p.requires_grad = False

    def unfreeze_top_layers(self, n=4):
        self.freeze_student()
        layers = self.student.encoder.layers
        for layer in layers[len(layers) - n:]:
            for p in layer.parameters(): p.requires_grad = True
        for p in self.student.encoder.layer_norm.parameters():
            p.requires_grad = True
        n_train = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'  🔓 Unfroze top {n} transformer layers | trainable: {n_train:,}')

    def set_distill_mode(self):
        for p in self.student.parameters():    p.requires_grad = True
        for p in self.teacher_sv.parameters(): p.requires_grad = False
        for p in self.teacher_df.parameters(): p.requires_grad = False
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'  📌 Distillation mode | trainable: {n:,}')

    def set_finetune_mode(self, n_layers=4):
        self.unfreeze_top_layers(n_layers)
        print(f'  📌 Fine-tune mode (top {n_layers} layers)')

    # ── Core forward ──────────────────────────────────────────────────────────
    def get_embedding(self, audio):
        """
        audio: (B, T) raw waveform
        returns: (B, LSTM_HIDDEN*2) — temporal summary via BiLSTM
        """
        frames = self.student(audio).last_hidden_state  # (B, T_f, 768)
        # BiLSTM: output (B, T_f, H*2), h_n (2*num_layers, B, H)
        _, (h_n, _) = self.lstm_agg(frames)
        # Concat final forward & backward hidden states
        # h_n shape: (num_layers*2, B, H)
        # Take last layer's forward ([-2]) and backward ([-1])
        h_fwd = h_n[-2]   # (B, H)
        h_bwd = h_n[-1]   # (B, H)
        return torch.cat([h_fwd, h_bwd], dim=-1)  # (B, H*2)

    def forward(self, audio):
        z_s = self.get_embedding(audio)         # (B, LSTM_OUT)

        with torch.no_grad():
            z_sv = self.teacher_sv(audio)       # (B, REDIMNET_DIM)
            z_df = self.teacher_df(audio)       # (B, ASSIST_DIM)

        s_sv = self.proj_sv(z_s)
        s_df = self.proj_df(z_s)

        loss_sv = 1 - F.cosine_similarity(s_sv, z_sv).mean()
        loss_df = 1 - F.cosine_similarity(s_df, z_df).mean()
        total   = ALPHA * loss_sv + BETA * loss_df

        return total, loss_sv.item(), loss_df.item()


model = DualDistillationModel(wav2vec, teacher_sv, teacher_df).to(DEVICE)

total_p   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model ready | Total: {total_p:,} | Trainable: {trainable:,}')
print(f'   LSTM output dim : {DualDistillationModel.LSTM_OUT}')

---
## ▶ CELL 10 — Build Optimizer & Resume

In [ ]:
scaler = GradScaler(enabled=USE_AMP)

def build_opt_sched(lr, total_epochs):
    opt   = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=WEIGHT_DECAY
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=total_epochs, eta_min=1e-6
    )
    return opt, sched

# Start in distill mode, build initial optimizer
model.set_distill_mode()
optimizer, scheduler = build_opt_sched(LR_DISTILL, DISTILL_EPOCHS)

print('\n🔍 Checking for checkpoint...')
start_epoch, current_phase, history = ckpt_mgr.resume(
    model, optimizer, scheduler, scaler
)

# If resuming into finetune phase, rebuild optimizer properly
if current_phase == 'finetune':
    model.set_finetune_mode(FINETUNE_LAYERS)
    optimizer, scheduler = build_opt_sched(LR_FINETUNE, FINETUNE_EPOCHS)
    print('  Rebuilt optimizer for fine-tune phase')

print(f'\n▶ Starting from epoch={start_epoch}, phase={current_phase}')

---
## ▶ CELL 11 — Chunked Training Engine

> **Memory loop per epoch**:
> `for each chunk → load into RAM → make DataLoader → train N batches → delete chunk → free_memory() → next chunk`
> 
> VRAM never holds more than one chunk worth of activations at a time.

In [ ]:
def train_on_chunk(chunk_tensors, optimizer, scaler):
    """
    Train on one pre-loaded chunk (list of tensors).
    Uses gradient accumulation across micro-batches.
    Returns (avg_total, avg_sv, avg_df).
    """
    dataset    = ChunkDataset(chunk_tensors)
    loader     = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=0,  # 0 = no subprocess → less RAM overhead
                            pin_memory=(DEVICE=='cuda'), drop_last=True)
    if len(loader) == 0:
        return 0.0, 0.0, 0.0

    model.train()
    total_l = sv_l = df_l = 0.0
    n_steps = 0
    optimizer.zero_grad(set_to_none=True)

    for step, audio in enumerate(loader):
        audio = audio.to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_AMP):
            loss, lsv, ldf = model(audio)
            loss = loss / ACCUM_STEPS   # scale for accumulation

        if USE_AMP:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        total_l += loss.item() * ACCUM_STEPS
        sv_l    += lsv
        df_l    += ldf
        n_steps += 1

        # Optimizer step every ACCUM_STEPS batches
        if (step + 1) % ACCUM_STEPS == 0:
            if USE_AMP:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()), GRAD_CLIP)
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

    return (total_l/n_steps if n_steps else 0,
            sv_l/n_steps    if n_steps else 0,
            df_l/n_steps    if n_steps else 0)


def run_phase(phase, start_ep, end_ep, lr):
    """
    Full training phase looping over epochs × chunks.
    Each chunk is loaded, trained, and deleted before the next.
    """
    global history, optimizer, scheduler

    if start_ep > end_ep:
        print(f'⏭  Phase [{phase}] already complete')
        return

    print(f'\n{"="*64}')
    print(f'  PHASE: {phase.upper()}  |  Epochs {start_ep}→{end_ep}  |  LR={lr}')
    print(f'  Chunks/epoch: {chunk_mgr.num_chunks()}  |  '
          f'Chunk size: {CHUNK_SIZE}  |  Batch: {BATCH_SIZE}×{ACCUM_STEPS}')
    print(f'{"="*64}')

    for epoch in range(start_ep, end_ep + 1):

        # ── Time guard ────────────────────────────────────────────────────────
        if should_stop_early():
            print(f'\n⏰ SESSION GUARD: < 25 min remaining!')
            ckpt_mgr.save(model, optimizer, scheduler, scaler,
                          epoch-1, phase, history, is_best=False)
            ckpt_mgr.push_to_hub(epoch-1, is_best=True)
            print(f'  ✅ Emergency save done. Re-run notebook to resume from epoch {epoch}.')
            return

        epoch_start = time.time()
        epoch_total = epoch_sv = epoch_df = 0.0
        n_chunks    = chunk_mgr.num_chunks()

        for c_idx in range(n_chunks):

            # Per-chunk time guard
            if should_stop_early(): break

            # 1. Load chunk into RAM
            chunk_data = chunk_mgr.load_chunk(c_idx)
            if len(chunk_data) < BATCH_SIZE:
                chunk_mgr.release_chunk()
                continue

            # 2. Train on this chunk
            t, sv, df = train_on_chunk(chunk_data, optimizer, scaler)

            epoch_total += t
            epoch_sv    += sv
            epoch_df    += df

            # 3. DELETE chunk data & free memory
            del chunk_data
            chunk_mgr.release_chunk()

            print(f'  Epoch {epoch:02d} | Chunk [{c_idx+1:03d}/{n_chunks}] '
                  f'loss={t:.4f} sv={sv:.4f} df={df:.4f} | {mem_status()}',
                  end='\r')

        print()  # newline after \r
        scheduler.step()

        avg_t  = epoch_total / n_chunks
        avg_sv = epoch_sv    / n_chunks
        avg_df = epoch_df    / n_chunks

        history['total'].append(avg_t)
        history['sv'].append(avg_sv)
        history['df'].append(avg_df)

        is_best = avg_t < ckpt_mgr.best_loss
        if is_best: ckpt_mgr.best_loss = avg_t

        elapsed = time.time() - epoch_start
        print(f'  ✦ Epoch [{epoch:02d}/{end_ep}] [{phase}]  '
              f'Total={avg_t:.4f}  SV={avg_sv:.4f}  DF={avg_df:.4f}  '
              f'LR={scheduler.get_last_lr()[0]:.1e}  t={elapsed:.0f}s '
              f'{"⭐" if is_best else ""}')
        print(f'  ⏱  {time_status()}')

        # Save locally + Drive
        ckpt_mgr.save(model, optimizer, scheduler, scaler,
                      epoch, phase, history, is_best=is_best)

        # Push to HF every N epochs
        if epoch % HF_PUSH_EVERY == 0 or epoch == end_ep:
            ckpt_mgr.push_to_hub(epoch, is_best=is_best)

    print(f'\n✅ Phase [{phase}] complete!')


print('✅ Training engine ready')

---
## ▶ CELL 12 — Phase 1: Distillation

In [ ]:
if current_phase == 'distill' and start_epoch <= DISTILL_EPOCHS:
    model.set_distill_mode()
    optimizer, scheduler = build_opt_sched(LR_DISTILL, DISTILL_EPOCHS)
    # Sync scheduler to resumed epoch
    for _ in range(start_epoch - 1): scheduler.step()

    run_phase('distill', start_epoch, DISTILL_EPOCHS, LR_DISTILL)
else:
    print('⏭  Distillation phase complete — skipping')

---
## ▶ CELL 13 — Phase 2: Fine-Tuning

In [ ]:
ft_start = start_epoch if current_phase == 'finetune' else 1

if ft_start <= FINETUNE_EPOCHS:
    print('🔓 Entering Fine-Tuning phase...')
    model.set_finetune_mode(FINETUNE_LAYERS)
    optimizer, scheduler = build_opt_sched(LR_FINETUNE, FINETUNE_EPOCHS)
    for _ in range(ft_start - 1): scheduler.step()
    ckpt_mgr.best_loss = float('inf')  # reset best tracking for FT

    run_phase('finetune', ft_start, FINETUNE_EPOCHS, LR_FINETUNE)
else:
    print('⏭  Fine-tuning phase complete — skipping')

---
## ▶ CELL 14 — Loss Curves

In [ ]:
def plot_history(history, distill_epochs, finetune_epochs):
    n = len(history['total'])
    if n == 0:
        print('No history to plot'); return

    x = list(range(1, n+1))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle('Dual-Teacher Distillation + Fine-Tuning (Colab)', fontsize=13)

    for ax, key, color, lbl in zip(
        axes,
        ['total','sv','df'],
        ['#2196F3','#4CAF50','#FF5722'],
        ['Total Loss','L_SV (ReDimNet)','L_DF (ASSIST)']
    ):
        ax.plot(x, history[key], color=color, lw=2, marker='o', ms=3)
        if distill_epochs < n:
            ax.axvline(x=distill_epochs+.5, color='gray',
                       ls='--', alpha=.6, label='FT start')
            ax.legend(fontsize=8)
        ax.set(title=lbl, xlabel='Epoch', ylabel='Cosine Distance')
        ax.grid(True, alpha=.3)
        ax.set_ylim(bottom=0)

    plt.tight_layout()
    out = '/content/loss_curves.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')

    # Also save to Drive
    try: shutil.copy(out, os.path.join(DRIVE_DIR, 'loss_curves.png'))
    except Exception: pass

    plt.show()
    print(f'✅ Saved: {out}')

plot_history(history, DISTILL_EPOCHS, FINETUNE_EPOCHS)

---
## ▶ CELL 15 — Final HF Push & Model Card

In [ ]:
# Push loss curves
for f, repo_name in [('/content/loss_curves.png','loss_curves.png')]:
    try:
        hf_api.upload_file(path_or_fileobj=f, path_in_repo=repo_name,
                           repo_id=HF_REPO_ID, repo_type='model',
                           commit_message='Loss curves')
    except Exception as e:
        print(f'⚠️  {e}')

# Model card
card = f"""---
tags: [audio, speaker-verification, deepfake-detection, knowledge-distillation, wav2vec2, lstm]
---
# Dual-Teacher Distilled Wav2Vec2 + LSTM

Student model: `facebook/wav2vec2-base` + Bidirectional LSTM aggregator,  
distilled from two specialized teachers.

| Teacher | Task | Model |
|---|---|---|
| Teacher SV | Speaker Verification | ReDimNet-B0 |
| Teacher DF | Deepfake Detection | ASSIST |

## Architecture
- Wav2Vec2 frame encoder → BiLSTM({LSTM_HIDDEN}×2) → proj heads
- LSTM captures temporal dynamics discarded by mean-pooling

## Training
- Platform: Google Colab | Chunked streaming ({CHUNK_SIZE} files/chunk)
- Phase 1: Distillation — {DISTILL_EPOCHS} epochs, LR={LR_DISTILL}
- Phase 2: Fine-tuning — {FINETUNE_EPOCHS} epochs (top {FINETUNE_LAYERS} layers), LR={LR_FINETUNE}
- Loss: `L = {ALPHA}*(1−cosine(proj_sv, z_sv)) + {BETA}*(1−cosine(proj_df, z_df))`
- Best loss: {ckpt_mgr.best_loss:.4f}

## Files
- `checkpoint_best.pt` — best checkpoint
- `checkpoint_latest.pt` — latest epoch
- `train_meta.json` — metadata
"""

readme = '/content/README.md'
with open(readme, 'w') as f: f.write(card)
try:
    hf_api.upload_file(path_or_fileobj=readme, path_in_repo='README.md',
                       repo_id=HF_REPO_ID, repo_type='model',
                       commit_message='Model card')
    print(f'✅ Model card: https://huggingface.co/{HF_REPO_ID}')
except Exception as e:
    print(f'⚠️  {e}')

---
## ▶ CELL 16 — Inference & Sanity Check

In [ ]:
def extract_embedding(audio_path_or_tensor):
    """
    Returns (LSTM_HIDDEN*2,) embedding — the distilled BiLSTM summary.
    Accepts file path or pre-loaded waveform tensor.
    """
    if isinstance(audio_path_or_tensor, str):
        w, sr = torchaudio.load(audio_path_or_tensor)
        w = w.mean(0)
        if sr != SAMPLE_RATE:
            w = torchaudio.functional.resample(w.unsqueeze(0), sr, SAMPLE_RATE).squeeze(0)
        if w.shape[0] < MAX_AUDIO_LEN: w = F.pad(w, (0, MAX_AUDIO_LEN - w.shape[0]))
        else: w = w[:MAX_AUDIO_LEN]
        w = w / (w.abs().max() + 1e-8)
    else:
        w = audio_path_or_tensor

    model.eval()
    with torch.no_grad(), autocast(enabled=USE_AMP):
        emb = model.get_embedding(w.unsqueeze(0).to(DEVICE))
    return emb.squeeze(0).cpu().float()


# Sanity check using first two tensors from chunk 0
print('Running sanity check...')
test_chunk = chunk_mgr.load_chunk(0)
if len(test_chunk) >= 2:
    e1 = extract_embedding(test_chunk[0])
    e2 = extract_embedding(test_chunk[1])
    e3 = extract_embedding(test_chunk[0])  # same as e1

    print(f'Embedding shape         : {e1.shape}  (BiLSTM output)')
    print(f'Cosine sim (same file)  : {F.cosine_similarity(e1.unsqueeze(0), e3.unsqueeze(0)).item():.4f}  ← ~1.0')
    print(f'Cosine sim (diff files) : {F.cosine_similarity(e1.unsqueeze(0), e2.unsqueeze(0)).item():.4f}')

del test_chunk; free_memory()
print('\n✅ Inference OK')

---
## ▶ CELL 17 — Summary

In [ ]:
ela = (time.time() - SESSION_START) / 3600
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

print('═'*64)
print('  TRAINING COMPLETE')
print('═'*64)
print(f'  GPU            : {gpu}')
print(f'  Runtime        : {ela:.2f} hours')
print(f'  Distill epochs : {DISTILL_EPOCHS}')
print(f'  Finetune epochs: {FINETUNE_EPOCHS}')
print(f'  Best loss      : {ckpt_mgr.best_loss:.4f}')
print(f'  HF repo        : https://huggingface.co/{HF_REPO_ID}')
print(f'  Drive backup   : {DRIVE_DIR}')
print('═'*64)
print()
print('  ▶ To resume next session:')
print('  1. Open this notebook in a new Colab runtime')
print('  2. Run all cells — CheckpointManager auto-loads from HF Hub')
print('  3. Training resumes from last saved epoch automatically')